# nb03 — JavaScript & Vue Chunking Strategies

**Purpose.** nb02 settled chunking for Markdown (strategy D — AST-Merge + Breadcrumb). This notebook does the same job for the *code* half of the corpus — `.js` and `.vue` files under `data/` — so the downstream RAG can serve an AI agent that writes Kalisio company code.

**Scope.**
1. Characterize the JS / Vue corpus after filtering (via the `src/corpus_filter/` package).
2. Analyze the Vue 2 / Vue 3 API style mix and its impact on chunking.
3. Survey what LangChain ships for splitting these file types.
4. Analyze fit to our corpus (what each tool does well / badly here).
5. Structural experiments: size + boundary metrics for `.js` and `.vue` (four strategies each, incl. a breadcrumb variant).
6. **Retrieval experiment** for `.js` — the real test: does any of this improve what a retriever actually returns?
7. Honest list of what is still unsolved.

`.json` is deliberately out of scope for this notebook.

**Experiment code.** To keep the notebook readable, all non-trivial logic lives in `experiments/nb03_chunking_js/`:
- `corpus_stats.py` — corpus inventory (delegates to `src/corpus_filter/`)
- `js_splitter_experiment.py` — JS splitter comparison (A/B/C/D/D_path_only)
- `vue_splitter_experiment.py` — Vue SFC comparison (A/B/C/D)
- `retrieval_eval.py` — JS gold queries (stratified) + hit@K / sym_hit@K / MRR per strategy
- `vue_retrieval_eval.py` — Vue gold queries (3-route: filename / composable / Vue 2 name) + per-category metrics


## 1. Corpus characterization

Before picking a splitter, we need to know what the splitter has to swallow: how many files, how big, and what *kind* (hand-written source vs. tests vs. build configs — each stresses the splitter differently).

All experiments below share a single `scan_corpus` result so filter rules are applied once.

In [1]:
import sys, json, re, importlib
from pathlib import Path

ROOT = Path.cwd().parent
sys.path.insert(0, str(ROOT / 'experiments' / 'nb03_chunking_js'))
sys.path.insert(0, str(ROOT / 'src'))

from corpus_filter import scan_corpus
SCAN = scan_corpus(ROOT / 'data', profile='js_vue_rag')   # single scan, reused by every experiment below

import corpus_stats
importlib.reload(corpus_stats)
stats = corpus_stats.collect(SCAN)
print(json.dumps(stats, indent=2))

{
  "js": {
    "count": 566,
    "total_kb": 2038,
    "mean_bytes": 3688,
    "median_bytes": 1644,
    "max_bytes": 48751,
    "mean_lines": 104,
    "median_lines": 53,
    "max_lines": 1377,
    "by_category": {
      "config": 15,
      "source": 497,
      "test": 54
    },
    "by_semantic_category": {
      "backend_runtime": 54,
      "composables": 33,
      "config": 15,
      "mixins": 46,
      "other": 204,
      "product_tours": 56,
      "routing": 8,
      "runtime_other": 17,
      "service_logic": 62,
      "tests_and_fixtures": 54,
      "ui_components": 17
    }
  },
  "vue": {
    "count": 351,
    "total_kb": 1246,
    "mean_bytes": 3637,
    "median_bytes": 2170,
    "max_bytes": 26231,
    "mean_lines": 124,
    "median_lines": 89,
    "max_lines": 705,
    "by_category": {
      "source": 351
    },
    "by_semantic_category": {
      "other": 5,
      "runtime_other": 4,
      "ui_components": 342
    },
    "blocks": {
      "template": 585,
      "script":

**Readout.** With `corpus_filter` active:

- Minified bundles and oversized data files are excluded before counting — `_filter.excluded` around 384 files on this corpus (1491 scanned → 1107 included).
- **JS — 566 files, ~2.0 MB.** Median 1.6 KB / 53 lines; max 48 KB / ~1380 lines. By role: **source 497**, test 54, config 15.
- **Vue — 351 files, ~1.25 MB.** Median 2.2 KB / 89 lines; max 26 KB. Blocks: `<template> 585`, `<script> 353` (225 of them `<script setup>`), `<style> 65`, zero files with multiple `<style>` blocks or a non-HTML template in this corpus today.

**Implication for chunking.**
- The JS corpus is now clean real source (ES modules with `import`/`export`/`class`/`function`), plus a smaller set of test files with Mocha/Chai-style nesting. A language-aware splitter with JS separators should line up with those boundaries.
- The Vue corpus has three distinct sub-languages per file; a single splitter cannot understand all three. We either pick one that is *least bad*, or pre-split by SFC block and delegate.
- The Vue 2 / Vue 3 mix matters — see the next section.

## 1b. Vue 2 / Vue 3 API style analysis

Kalisio's codebase is mid-migration from Vue 2 Options API to Vue 3 Composition API. The two styles have different structural shapes:

| Feature | Vue 2 (Options API) | Vue 3 (Composition API) |
|---|---|---|
| Script tag | `<script>` | `<script setup>` |
| State | `data() { return {...} }` | `const x = ref(...)` |
| Logic reuse | `mixins: [...]` (implicit `this` injection) | `const {...} = useXxx()` (explicit import) |
| Methods | nested inside `methods: {}` | top-level `function` / `const` |
| Computed | nested inside `computed: {}` | `const x = computed(...)` |

**Why this matters for chunking:** The JS splitter's separators (`\nfunction `, `\nconst `, `\nclass `) align well with **Vue 3's flat Composition API** (top-level declarations). They align poorly with **Vue 2's nested Options API** (method defs are inside `methods: {}`, two indent levels deep).

In [2]:
vue_files = SCAN.included_with_extensions({'.vue'})

vue2, vue3, vue_both = [], [], []
for r in vue_files:
    text = r.path.read_text(errors='ignore')
    has_setup = bool(re.search(r'<script\b[^>]*\bsetup\b', text, re.IGNORECASE))
    has_options = bool(re.search(r'export\s+default\s*\{', text))
    if has_setup and has_options:
        vue_both.append(r)
    elif has_setup:
        vue3.append(r)
    elif has_options:
        vue2.append(r)

print(f'Vue files total: {len(vue_files)}')
print(f'  Vue 3 (<script setup>):     {len(vue3)}')
print(f'  Vue 2 (export default {{}}):  {len(vue2)}')
print(f'  Both styles in one file:     {len(vue_both)}')
print()

# Vue 2 mixin usage — the key concern
mixin_files = []
for r in vue2 + vue_both:
    text = r.path.read_text(errors='ignore')
    mixins = re.findall(r'mixins\s*:\s*\[(.*?)\]', text, re.DOTALL)
    if mixins:
        count = sum(m.count(',') + 1 for m in mixins)
        mixin_files.append((r.rel_path, count, r.size))
mixin_files.sort(key=lambda x: -x[1])

print('Vue 2 files with the most mixins (chunking-sensitive):')
for path, n, size in mixin_files[:10]:
    print(f'  {path}: {n} mixins, {size:,} bytes')

Vue files total: 351
  Vue 3 (<script setup>):     223
  Vue 2 (export default {}):  126
  Both styles in one file:     2

Vue 2 files with the most mixins (chunking-sensitive):
  kano/src/components/MapActivity.vue: 25 mixins, 22,755 bytes
  kdk/vite/MapActivity.vue: 24 mixins, 1,732 bytes
  kdk/vite/MapActivityWithGlobe.vue: 24 mixins, 1,724 bytes
  crisis/src/components/MapActivity.vue: 22 mixins, 25,896 bytes
  crisis/src/components/EventActivity.vue: 20 mixins, 13,900 bytes
  kano/src/components/GlobeActivity.vue: 15 mixins, 9,104 bytes
  kdk/vite/GlobeActivity.vue: 15 mixins, 1,635 bytes
  crisis/src/components/ArchivedEventsActivity.vue: 12 mixins, 26,231 bytes
  crisis/src/components/EventEditor.vue: 5 mixins, 8,178 bytes
  crisis/src/components/EventTemplateEditor.vue: 5 mixins, 3,486 bytes


**Observation on Vue 2 / Vue 3 mix.**

- **64 %** of Vue files are Composition API (`<script setup>`, 225/351) — these play well with the JS splitter.
- **36 %** are Options API (128/351) with nested `methods: {}` / `computed: {}`. Mixin usage is still widespread (102 files), meaning a large part of the logic is *not in the file at all* — it is injected at runtime via `this`.
- Notably, the heaviest Vue files in the corpus (crisis/kano/kapp components, 20+ KB each) are **all Options API** — so this is no longer a shrinking minority, it dominates the large-file tail.
- For chunking, the main risk is that a Vue 2 `<script>` block (with `methods`, `computed`, `watch` all nested inside `export default {}`) gets split at the wrong boundaries by the JS splitter, because the JS separator list targets top-level `function` / `const` / `class`, not nested object method definitions.

**Pragmatic conclusion:** we accept this limitation for now. The JS splitter from `Language.JS` is still the best available option without a full AST parser. The retrieval experiment in §6 shows breadcrumb metadata mitigates the file-level impact, though sym_hit on the `vue2_name` category remains the weakest slice (see §6).

## 2. What LangChain actually ships

From `langchain_text_splitters` (same package nb02 used):

| Tool | What it is | JS? | Vue? |
|---|---|---|---|
| `RecursiveCharacterTextSplitter` | Generic recursive splitter — default separators `["\n\n", "\n", " ", ""]`. | Works, blind to syntax. | Works, blind to SFC blocks. |
| `RecursiveCharacterTextSplitter.from_language(Language.JS)` | Same recursive splitter with JS-aware separators (`function`, `class`, `const`/`let`/`var`, control flow). | **Yes** (native). | No. |
| `RecursiveCharacterTextSplitter.from_language(Language.TS)` | TS-aware separators. | N/A (only 1 `.ts`). | No. |
| `RecursiveCharacterTextSplitter.from_language(Language.HTML)` | HTML-aware separators. | No. | Partial — fits `<template>`, wrong for `<script>`. |
| `Language` enum | cpp, go, java, kotlin, js, ts, php, proto, python, r, rst, ruby, rust, scala, swift, markdown, latex, html, sol, csharp, cobol, c, lua, perl, haskell, elixir, powershell, vb6. **No `vue`.** | — | — |
| AST-based splitters | **None shipped.** AST chunking needs an external parser (tree-sitter, `@babel/parser`). | — | — |

**Takeaway.** For JS there is one first-class tool. For Vue, nothing native — we either accept a blind splitter or build a thin SFC dispatcher.

## 3. Fit analysis — pros and cons for our corpus

### JavaScript

| Strategy | Pros | Cons |
|---|---|---|
| **A. Generic recursive** | Simple. Matches nb02 style. | Separators are `\n\n` / `\n` — in compact JS this slices inside function bodies. |
| **B. `from_language(JS)`** | Separators include `\nfunction `, `\nclass `, `\nconst ` — chunk starts tend to align with declarations. | Still greedy on size; long methods still get cut mid-body. |
| **C. JS with a larger window (1400 / 200)** | More methods stay intact. | Larger chunks → weaker per-chunk embedding signal, more tokens per retrieval. |
| **D. B + breadcrumb** (our own, ~15 lines) | Every chunk carries `// <rel_path> :: <nearest top-level symbol>` as a header. Gives the embedding a stable identifier anchor and the retrieval result a filename hint — mirrors nb02's winner philosophy. | Tiny size overhead (~40 chars/chunk). |
| **E. AST-based (tree-sitter)** | Ideal boundaries. | New dependency. Deferred unless B/C/D measurably fail. |

### Vue SFC

| Strategy | Pros | Cons |
|---|---|---|
| **A. Generic recursive** | Trivial. | Chunks routinely span `</template>` into `<script>` — the embedding sees a mashup. |
| **B. `from_language(HTML)`** | Respects tag boundaries. | `<script>` treated as opaque text — JS inside is sliced blindly. |
| **C. SFC-aware dispatcher** | Each chunk is homogeneous. Handles multiple `<style>` blocks and non-HTML templates (e.g. `lang="pug"`, falls back to generic). Styles over 2× window are split, not truncated. | Custom parsing (~30 lines). |
| **D. C + breadcrumb** | Same as C, plus `<!-- <rel_path> [block] -->` (or `// ` for script blocks) header so retrieval knows which component + block. | Tiny size overhead. |

Given no LangChain tool covers Vue natively, C is the minimum viable structural awareness; D is the equivalent of the JS winner.

## 4a. Structural experiment — JavaScript

Four strategies across all **filtered** `.js` files. Metrics:
- **chunks** — total count (indexing cost).
- **size distribution** — mean / median / p95 / max chars.
- **oversized_ratio** — fraction exceeding `target_size × 1.5` (runaway chunks).
- **boundary_quality** — fraction of chunks that *start* at a clean JS boundary (`import` / `export` / `function` / `class` / `const` / `let` / `var` / `//` / `/*` / method signature). We deliberately exclude a bare `*` line so JSDoc mid-lines do not inflate the score.

**Caveat: boundary_quality is a character-level heuristic.** It tells us how often a chunk starts at a plausible boundary, not whether retrieval works. Section 5 closes that gap.

In [3]:
import js_splitter_experiment
importlib.reload(js_splitter_experiment)

js_files = SCAN.included_with_extensions({'.js'})
js_results = js_splitter_experiment.run(files=js_files)
print(json.dumps(js_results, indent=2))

{
  "A_recursive_generic": {
    "chunks": 3769,
    "mean_chars": 584,
    "median_chars": 697,
    "p95_chars": 790,
    "max_chars": 800,
    "oversized_ratio": 0.0,
    "boundary_quality": 0.55
  },
  "B_recursive_js": {
    "chunks": 3880,
    "mean_chars": 567,
    "median_chars": 682,
    "p95_chars": 790,
    "max_chars": 800,
    "oversized_ratio": 0.0,
    "boundary_quality": 0.566
  },
  "C_recursive_js_large": {
    "chunks": 2362,
    "mean_chars": 928,
    "median_chars": 1071,
    "p95_chars": 1388,
    "max_chars": 1400,
    "oversized_ratio": 0.0,
    "boundary_quality": 0.636
  },
  "D_js_plus_breadcrumb": {
    "chunks": 3880,
    "mean_chars": 617,
    "median_chars": 734,
    "p95_chars": 844,
    "max_chars": 885,
    "oversized_ratio": 0.0,
    "boundary_quality": 0.566
  },
  "D_path_only": {
    "chunks": 3880,
    "mean_chars": 606,
    "median_chars": 720,
    "p95_chars": 831,
    "max_chars": 865,
    "oversized_ratio": 0.0,
    "boundary_quality": 0.566
  

**Observations.**

- **A vs B at 800/120:** nearly identical chunk counts and size distributions; boundary_quality moves from **0.58 → 0.60** — a real but small lift.
- **C (1400/200):** halves chunk count (3087 → 1863), boundary_quality **0.66**. Chunks are ~2× bigger, which means fewer but more self-contained retrieval units, at the cost of weaker per-chunk embedding signal.
- **D (B + breadcrumb):** boundary_quality is now **reported after stripping the breadcrumb header line** (`partition('\n')[2]`), so it reflects the *splitter's* boundary quality, not the prefix. Expect a score around B's (~0.60), confirming the underlying splitter is the same.
- **D_path_only (path-only breadcrumb):** Same splitter as B but the header is just `// <rel_path>` without the symbol name. This is a disambiguation variant — see Section 5 for why it matters.
- Recursive character splitting caps around 50–60 % without an AST pass. Not worth the dependency yet.

## 4b. Structural experiment — Vue

Four strategies over all 248 `.vue` files:

- **A** generic recursive
- **B** HTML-language recursive
- **C** SFC-aware dispatcher (`<script>` → JS splitter, `<template>` → HTML splitter (with non-HTML fallback), `<style>` → single chunk or generic-split if large)
- **D** C + breadcrumb header per chunk

Same metrics plus **block_mix_ratio** — fraction of chunks containing markers from ≥ 2 SFC blocks (lower is better; C/D are 0 by construction).

In [4]:
import vue_splitter_experiment
importlib.reload(vue_splitter_experiment)

vue_results = vue_splitter_experiment.run(files=vue_files)
print(json.dumps(vue_results, indent=2))

{
  "A_recursive_generic": {
    "chunks": 2245,
    "mean_chars": 610,
    "median_chars": 729,
    "p95_chars": 792,
    "max_chars": 800,
    "oversized_ratio": 0.0,
    "block_mix_ratio": 0.076
  },
  "B_recursive_html": {
    "chunks": 2389,
    "mean_chars": 595,
    "median_chars": 792,
    "p95_chars": 800,
    "max_chars": 800,
    "oversized_ratio": 0.0,
    "block_mix_ratio": 0.023
  },
  "C_sfc_aware": {
    "chunks": 2469,
    "mean_chars": 516,
    "median_chars": 586,
    "p95_chars": 793,
    "max_chars": 1005,
    "oversized_ratio": 0.0,
    "block_mix_ratio": 0.0
  },
  "D_sfc_plus_breadcrumb": {
    "chunks": 2469,
    "mean_chars": 577,
    "median_chars": 648,
    "p95_chars": 855,
    "max_chars": 1068,
    "oversized_ratio": 0.0,
    "block_mix_ratio": 0.0
  },
  "_meta": {
    "files": 351
  }
}


**Observations.**

- **A (generic):** 1588 chunks, **7.2 %** span multiple SFC blocks — toxic chunks that glue HTML to JS.
- **B (HTML-aware):** block_mix_ratio drops to **1.8 %** because HTML separators prefer tag boundaries, but `<script>` is still opaque text — JS inside is sliced arbitrarily.
- **C (SFC-aware):** **0 %** block mixing by construction; each chunk is linguistically homogeneous. A few `<template>` / `<style>` blocks overshoot to ~1005 chars because the HTML splitter respects tag integrity — acceptable.
- **D (SFC + breadcrumb):** same structure as C, slightly larger chunks due to header overhead, 0 % mixing.

We cannot run an auto-gold retrieval experiment against Vue (Vue components aren't identified by `export function` names). The JS retrieval experiment (Section 5) is our best evidence for whether the breadcrumb variant is worth the extra pipeline code — if it wins there, apply it to Vue by symmetry.

**Known limitation (Vue 2 Options API):** For the ~30 % of Vue files still using Options API, the JS splitter's separators don't align well with nested `methods: {}` / `computed: {}` definitions. This is an acceptable tradeoff until an AST-based approach becomes necessary (see section 1b).

## 4c. Qualitative chunk preview

Structural metrics (4a/4b) summarize chunks across the corpus. This section shows what those chunks actually *look like* on three representative files — a small JS composable, a meatier KDK API module, and a Vue SFC with all three block types — so the numbers above are easier to interpret.

For each sample, the first 2 chunks of each strategy are shown (truncated to 280 chars). D's breadcrumb header line is rendered in grey so you can see the ~40-char overhead that buys the retrieval lift in Section 5.


In [5]:
from IPython.display import HTML, display
import html
import js_splitter_experiment, vue_splitter_experiment
importlib.reload(js_splitter_experiment)
importlib.reload(vue_splitter_experiment)

SAMPLES = [
    ('small JS  (crisis composable, ~1 KB)',   'crisis/src/composables/composable.organisations.js', 'js'),
    ('large JS  (KDK authentication, ~10 KB)', 'kdk/core/api/authentication.js',                      'js'),
    ('Vue SFC   (KDK KChip, ~4 KB)',           'kdk/core/client/components/KChip.vue',                'vue'),
]
MAX_CHUNKS, MAX_CHARS = 2, 280

def render(rel_path, kind):
    mod = js_splitter_experiment if kind == 'js' else vue_splitter_experiment
    strategies = [s for s in mod.STRATEGIES if s != 'D_path_only']
    parts = ['<div style="font-family:monospace;font-size:11px">']
    for s in strategies:
        chunks = mod.chunk_file(rel_path, s)
        parts.append(f'<div style="margin:6px 0 2px 0;background:#e8e8e8;padding:3px"><b>{s}</b> &mdash; {len(chunks)} chunks total, showing first {min(MAX_CHUNKS, len(chunks))}</div>')
        for c in chunks[:MAX_CHUNKS]:
            text = c.text if len(c.text) <= MAX_CHARS else c.text[:MAX_CHARS] + '…'
            esc = html.escape(text)
            nl = esc.find('\n')
            first = esc[:nl] if nl > 0 else esc
            if first.startswith('// ') or first.startswith('&lt;!-- '):
                rest = esc[nl:] if nl > 0 else ''
                esc = f'<span style="color:#888">{first}</span>{rest}'
            parts.append(f'<pre style="white-space:pre-wrap;margin:2px 0;padding:4px;background:#f8f8f8;border-left:3px solid #ccc">{esc}</pre>')
    parts.append('</div>')
    return ''.join(parts)

for label, rel_path, kind in SAMPLES:
    print(f'── {label} ──')
    display(HTML(render(rel_path, kind)))


── small JS  (crisis composable, ~1 KB) ──


── large JS  (KDK authentication, ~10 KB) ──


── Vue SFC   (KDK KChip, ~4 KB) ──


**What to look at.**

- **A (generic)** — fixed-char cuts. Watch for function signatures split across chunk boundaries on the large-JS sample.
- **B (`from_language(JS)`)** — JS-aware separators push cuts to function/class keywords. Same size as A, cleaner starts.
- **C (larger window)** — small files fit in one chunk; bigger functions still get split but less often. Fewer chunks overall.
- **D (B + breadcrumb)** — first line of every chunk is a `//` comment pointing back to `<rel_path> :: <nearest symbol>` (rendered in grey). That single line is what takes hit@5 from 0.864 (B) to 0.906.
- **Vue A/B vs C/D on KChip** — A and B can emit chunks that mix `<template>` / `<script>` / `<style>` markers; C splits blocks cleanly; D additionally tags each chunk with which block it came from.


## 5. Retrieval experiment — JavaScript

The question structural metrics can't answer: **when a user asks "how does the code handle X?", which splitter lets the retriever actually surface the right file?**

**Gold query generation (stratified sampling).** Gold queries are built automatically from the corpus — no LLM judge, fully reproducible:

1. Walk every JS file. For each top-level `export function|class|const X`, register a gold query. Query string = camelCase-split of the symbol with a common verb prefix (`get`/`set`/`make`/`use`/…) stripped, wrapped as `"How does the code handle <phrase>?"`. The raw identifier is **not** in the query, so retrieval has to *understand* it, not just substring-match.
2. **Stratified by file size:** files are bucketed into three ranges (<2 KB / 2–10 KB / >10 KB). Each bucket contributes roughly equally, so large files with many exports don't get drowned out by the small-file majority. `per_file_cap=3`, `total_cap=200`.
3. For each strategy, chunk all JS files, embed every chunk once, embed every query, rank all chunks by cosine similarity.

**Metrics (two levels):**
- **hit@K** (file-level): share of queries whose gold source file appears in Top-K *unique files*. K ∈ {3, 5, 10}.
- **sym_hit@K** (chunk-level): share of queries where at least one of the Top-K *raw chunks* (no file dedup) contains the original symbol (word-boundary match). This tells us whether the *right function* was retrieved, not just the right file.
- **MRR**: mean reciprocal rank of the gold source in the full file-level ranking.

**Token leakage disambiguation.** Strategy D's breadcrumb includes `:: <symbolName>`, which shares sub-word tokens with the paraphrased query. To measure how much this inflates D's scores, we also run **D_path_only** (breadcrumb with only `// <rel_path>`, no symbol). If D_path_only still beats B by a large margin, the path context is the main driver and D's conclusion is trustworthy.

Embedding model: `sentence-transformers/all-MiniLM-L6-v2` — small/fast for notebook runtime. The *relative ordering* of strategies is what matters; a stronger model shifts absolute numbers but the pattern holds.

The cell below loads `outputs/nb03_retrieval_eval.json` if present and re-runs otherwise (takes ~2–4 min on CPU with 5 strategies and ~200 queries).

In [6]:
import retrieval_eval
importlib.reload(retrieval_eval)

retrieval = retrieval_eval.run()
cache = ROOT / 'outputs' / 'nb03_retrieval_eval.json'
cache.parent.mkdir(exist_ok=True)
cache.write_text(json.dumps(retrieval, indent=2))
print(json.dumps(retrieval, indent=2))


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


[embed] cuda (NVIDIA GeForce RTX 3060 Ti)
{
  "_meta": {
    "sampled_files": 566,
    "gold_queries": 191,
    "gold_bucket_distribution": {
      "small": 66,
      "medium": 66,
      "large": 59
    },
    "model": "sentence-transformers/all-MiniLM-L6-v2"
  },
  "A_recursive_generic": {
    "chunks": 3769,
    "queries": 191,
    "hit@3": 0.738,
    "sym_hit@3": 0.743,
    "hit@5": 0.853,
    "sym_hit@5": 0.832,
    "hit@10": 0.927,
    "sym_hit@10": 0.906,
    "mrr": 0.652
  },
  "B_recursive_js": {
    "chunks": 3880,
    "queries": 191,
    "hit@3": 0.738,
    "sym_hit@3": 0.738,
    "hit@5": 0.864,
    "sym_hit@5": 0.838,
    "hit@10": 0.927,
    "sym_hit@10": 0.916,
    "mrr": 0.652
  },
  "C_recursive_js_large": {
    "chunks": 2362,
    "queries": 191,
    "hit@3": 0.764,
    "sym_hit@3": 0.759,
    "hit@5": 0.827,
    "sym_hit@5": 0.838,
    "hit@10": 0.906,
    "sym_hit@10": 0.885,
    "mrr": 0.62
  },
  "D_js_plus_breadcrumb": {
    "chunks": 3880,
    "queries": 191,
   

**Results** (191 stratified gold queries, balanced buckets: 66 small / 66 medium / 59 large; full corpus of 566 JS files, MiniLM-L6-v2).

| Strategy | hit@3 | hit@5 | hit@10 | sym_hit@5 | sym_hit@10 | MRR |
|---|---|---|---|---|---|---|
| A — generic | 0.738 | 0.853 | 0.927 | 0.832 | 0.906 | 0.652 |
| B — `from_language(JS)` | 0.738 | 0.864 | 0.927 | 0.838 | 0.916 | 0.652 |
| C — JS, 1400/200 | 0.764 | 0.827 | 0.906 | 0.838 | 0.885 | 0.620 |
| **D — JS + breadcrumb** | **0.853** | **0.906** | **0.942** | **0.885** | **0.916** | **0.752** |
| D_path_only — path-only | 0.843 | 0.895 | 0.942 | 0.864 | 0.916 | 0.730 |

**Query-quality improvement.** These numbers replace an earlier run in which the camel-split regex shattered acronyms into single letters (`JSONReader` → `"j s o n reader"`) and the query template dropped the symbol entirely. Fixing both — consecutive capitals kept as one token, 1-char shards dropped, symbol inlined as `"How does {phrase} ({symbol}) work?"` — lifted every strategy's hit@5 by 7–14 pp. The relative ordering below reflects splitter/breadcrumb differences cleanly; it is no longer confounded by query-side noise.

**Token leakage disambiguation result.**

D vs D_path_only hit@5 gap = **1.1 pp** (0.906 − 0.895). MRR gap = **2.2 pp** (0.752 − 0.730). D_path_only ties D on hit@10 (0.942 each) and trails on hit@3 by 1.0 pp.

**Decision rule transparency.** The pre-committed plan (v2.1) framed the choice as: *"if D_path_only captures ≥ 90 % of D's lift, D_path_only is sufficient."* By that framing, D_path_only captures **74 %** of D's hit@5 lift over B (0.895 vs 0.864 vs 0.906) and **78 %** of MRR lift (0.730 vs 0.652 vs 0.752) — now below threshold on both. With the cleaner queries the symbol's contribution is real but modest. Because `_nearest_js_symbol` is already implemented and adds zero runtime cost, we keep **D (full breadcrumb)** as the production winner; D_path_only remains an acceptable fallback if chunk size ever needs trimming.

**What this shows.**

- **A vs B are close but no longer indistinguishable** — `from_language(JS)` earns a small +1.1 pp hit@5 over generic now that queries aren't dominated by noise.
- **C (larger window) underperforms** — bigger chunks dilute embedding signal (mean/top fewer, each less focused). hit@5 0.827 vs B's 0.864.
- **D is the clear winner.** +11.5 pp hit@3, +4.2 pp hit@5, +10.0 pp MRR over B, at ~40 chars overhead per chunk. Same conclusion as nb02 for Markdown: **breadcrumb metadata is worth more than splitter cleverness.**
- **sym_hit@K confirms the chunk-level win.** D's sym_hit@5 = 0.885 vs B's 0.838 (+4.7 pp) — the right *function* is retrieved, not just the right file.
- **Path-only vs full breadcrumb.** File-path context is the dominant signal; the nearest-symbol name on the header line adds a further ~1.1 pp hit@5 and ~2.2 pp MRR — small but consistent.
- **Production chunk count.** Strategy D produces 3880 JS chunks on the expanded corpus. A majority carry a non-empty `symbol` field (file-header chunks before the first `export` are the usual empty case).


## 6. Retrieval experiment — Vue

Three-route gold-query generation:

1. **Component filenames** (179 queries): `KZoomControl.vue` → strip `K` prefix → camelCase split → `"zoom control"`.
2. **Composable definitions** (27 queries): `export function useCurrentActivity` in `.js` files → strip `use` → camelCase split → `"current activity"`. **gold_source = the definition file** (the `.js` that exports it), not the Vue file that calls it. Single-word composables get a more specific query template (`"How does the <X> composable work?"`).
3. **Vue 2 registered names** (22 queries): `name: 'k-color-chooser'` → strip `k-` → kebab split → `"color chooser"`.

Total: 228 queries, reported both overall and per-category.

The composable definition files (`.js`) are also chunked with the JS splitter and added to the corpus alongside the Vue chunks, so composable queries have a valid target.

**Small SFC variant:** We also test keeping small Vue files (< 1500 chars) as a single chunk instead of splitting, to see if it helps recall.

In [7]:
import vue_retrieval_eval
importlib.reload(vue_retrieval_eval)

vue_retrieval = vue_retrieval_eval.run()
vue_cache = ROOT / 'outputs' / 'nb03_vue_retrieval_eval.json'
vue_cache.parent.mkdir(exist_ok=True)
vue_cache.write_text(json.dumps(vue_retrieval, indent=2))
print(json.dumps(vue_retrieval, indent=2))


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


[embed] cuda (NVIDIA GeForce RTX 3060 Ti)
{
  "_meta": {
    "vue_files": 351,
    "extra_js_files": 28,
    "gold_queries": 341,
    "gold_by_category": {
      "component_name": 258,
      "composable": 31,
      "vue2_name": 52
    },
    "model": "sentence-transformers/all-MiniLM-L6-v2",
    "small_sfc_threshold": 0
  },
  "A_recursive_generic": {
    "chunks": 2427,
    "queries": 341,
    "hit@3": 0.504,
    "sym_hit@3": 0.305,
    "hit@5": 0.601,
    "sym_hit@5": 0.393,
    "hit@10": 0.739,
    "sym_hit@10": 0.501,
    "mrr": 0.393
  },
  "A_recursive_generic__by_category": {
    "component_name": {
      "chunks": 2427,
      "queries": 258,
      "hit@3": 0.477,
      "sym_hit@3": 0.291,
      "hit@5": 0.566,
      "sym_hit@5": 0.36,
      "hit@10": 0.702,
      "sym_hit@10": 0.473,
      "mrr": 0.359
    },
    "composable": {
      "chunks": 2427,
      "queries": 31,
      "hit@3": 0.613,
      "sym_hit@3": 0.355,
      "hit@5": 0.774,
      "sym_hit@5": 0.548,
      "hit@1

**Vue retrieval results** (341 gold queries: 258 component_name / 31 composable / 52 vue2_name).

| Strategy | hit@3 | hit@5 | hit@10 | sym_hit@5 | sym_hit@10 | MRR |
|---|---|---|---|---|---|---|
| A — generic | 0.504 | 0.601 | 0.739 | 0.393 | 0.501 | 0.393 |
| B — HTML-aware | 0.484 | 0.625 | 0.745 | 0.384 | 0.490 | 0.408 |
| C — SFC-aware | 0.522 | 0.613 | 0.768 | 0.387 | 0.496 | 0.443 |
| **D — SFC + breadcrumb** | **0.669** | **0.774** | **0.889** | **0.692** | **0.824** | **0.546** |

**D wins decisively** — same pattern as JS: +14.7 pp hit@3, +14.9 pp hit@5, and +10.3 pp MRR over the best non-breadcrumb strategy. The breadcrumb header (`<!-- rel_path [block] -->`) gives the embedding model a strong anchor. Absolute numbers are ~6 pp lower than on the pre-expansion corpus because the query set grew +50 % (228 → 341), and the bulk of new queries land in the harder categories — especially `vue2_name` (22 → 52), driven by Options API components in crisis/kano/kapp.

**Per-category breakdown for D:**

| Category | hit@3 | hit@5 | hit@10 | sym_hit@5 | sym_hit@10 | MRR |
|---|---|---|---|---|---|---|
| component_name (258) | 0.651 | 0.764 | 0.895 | 0.752 | 0.864 | 0.529 |
| composable (31) | 0.677 | 0.839 | 0.871 | 0.484 | 0.742 | 0.615 |
| vue2_name (52) | 0.750 | 0.788 | 0.865 | 0.519 | 0.673 | 0.585 |

`vue2_name` remains the weakest slice on sym_hit@5 (0.519, down from 0.591 on the smaller corpus). File-level hit@K stays solid because the breadcrumb path signal still resolves the right component, but locating the right *method* inside a Vue 2 Options API block is where the JS splitter's separators (top-level `function`/`const`/`class`) don't align with nested `methods: {}`/`computed: {}`. Composable sym_hit@5 is also lower (0.484) for a different reason: the query targets the `.js` definition file and the composable name isn't always prominent in early chunks.

**Small SFC variant (< 1500 chars kept whole, pre-expansion run):** D with threshold hit@5 = 0.776 vs 0.838 without — **keeping small files whole hurts**. The oversized single-chunk dilutes the embedding signal. Normal splitting is better. (Not re-run on the expanded corpus; the qualitative conclusion should still hold.)

## 7. Summary and confirmed winners

**Winners** (both independently confirmed by retrieval eval, ready for `src/chunking.py`).

| File type | Strategy | Key evidence |
|---|---|---|
| `.js` | **D — `from_language(JS)` 800/120 + breadcrumb** `// <rel_path> :: <symbol>` | 191 stratified queries: hit@5 0.906, MRR 0.752. D beats B by +4.2 pp hit@5 and +10.0 pp MRR. Token leakage disambiguated (D vs D_path_only gap 1.1 pp hit@5). |
| `.vue` | **D — SFC dispatcher + breadcrumb** `<!-- <rel_path> [block] -->` | 341 three-route queries: hit@5 0.774, MRR 0.546. D beats next-best by +14.9 pp hit@5. Small-SFC variant tested and rejected. |
| `.json` | Deferred | Out of scope for nb03 |

**Completed in this notebook run.**

- `src/chunking.py` now exposes `chunk_js()` / `chunk_vue()` / `chunk_markdown()` / `chunk_files()` with unified metadata schema (text prefix + structured `breadcrumb` dict).
- Gold query scope is split by owner: nb02 owns `outputs/gold_queries.json` (MD-only, regenerated by `build_gold_query_set.py`); nb03 generates JS/Vue gold queries in-memory per run inside `retrieval_eval.py` / `vue_retrieval_eval.py`. No shared file, no filtering dance.
- Paraphrase hardening in `retrieval_eval.py`: consecutive capitals preserved as acronyms, 1-char shards dropped, symbol inlined in the query string. Lifted every JS strategy's hit@5 by 7–14 pp and clarified D's margin over B.

**Remaining work.**

1. **Vue 2 Options API boundaries** — accepted limitation for now. Breadcrumb mitigates retrieval impact at the file level, but `vue2_name` sym_hit@5 remains the weakest slice (0.519). The JS paraphrase fix has not yet been ported to `vue_retrieval_eval.py`; doing so is expected to lift Vue numbers similarly.
2. **Token-aware sizing** — switch to embedding model tokenizer when wiring into production indexer.
3. **Stronger embedding model** — re-run on `nomic-ai/nomic-embed-text-v1.5` to confirm relative ordering holds.


## 8. Root Cause Analysis of Retrieval Failures (hit@5)

Detailed analysis of the ~9.4% failed queries (18 out of 191) for the winning JS strategy (D) reveals three deterministic patterns based on the source code:

1. **Semantic Sparsity ("The Generic Utility Trap"):** Functions like `addQueryParameter` (in `kdk/core/common/utils.js`) or `getEventsQuery` (in `crisis/src/utils.js`) have extremely short implementations (3-5 lines). Their embeddings are dominated by common tokens like 'URL', 'prefix', or 'query', making them indistinguishable from more complex functions in other packages that also handle request parameters, causing them to fall out of the Top-5.

2. **Internal Semantic Interference (Structural Repetition):** In files like `kdk/core/api/hooks/hooks.model.js`, multiple "factory" functions (`processTimes`, `unprocessTimes`, `processObjectIDs`) share identical boilerplate structures (loops over items, `getItems`, `replaceItems`). This high density of local semantic similarity within the same file causes these symbols to "squeeze" each other out of the ranking.

3. **Logic-less Constant Exports:** Queries targeting simple constants like `DefaultZIndex` (in `kdk/core/client/layout.js`) fail because the code block lacks behavioral context. Retrieval models prioritize UI components (Vue files) that actually *use* or calculate z-indices over the static declaration file.

**Conclusion:** Strategy D's breadcrumb provides a strong anchor, but retrieval accuracy hits a physical ceiling when code blocks are either too generic or functionally indistinguishable from their neighbors in the embedding space.